In [0]:
schema_str="cid int, fname string, lname string, age int, prof string"
cust_raw_df=spark.read.csv("/Volumes/wh_gb_test/gb_test/my_volume/WH_DATA/empdata_NAM_20260620.csv",header=True,sep=",",schema=schema_str,mode="dropmalformed")

cust_raw_df.show(10)

In [0]:
stndrd_df_1 =cust_raw_df.selectExpr("cid as Customer_id","  fname || ' ' || lname AS Full_name","age","prof","current_date as LOAD_DATE","current_user() as created_by")
stndrd_df_1.show()

In [0]:
files = dbutils.fs.ls('/Volumes/wh_gb_test/gb_test/my_volume/WH_DATA/')

In [0]:
%sql
SELECT
    _metadata.file_name         AS source_file_name
FROM read_files(
    '/Volumes/wh_gb_test/gb_test/my_volume/WH_DATA/*20260620*',
    format => 'csv',
    header => 'true'
)

In [0]:
# Just file names
from pyspark.sql.functions import col, to_date, regexp_extract
df_files = (spark.read
    .format("csv")
    .option("header", "true")
    .load("/Volumes/wh_gb_test/gb_test/my_volume/WH_DATA/*20260620*")
    .select(
        col("_metadata.file_name").alias("source_file_name"), to_date(regexp_extract(col("_metadata.file_name"), r"(\d{8})", 1),             "yyyyMMdd").alias("data_date")
    ) .distinct()
)


df_files.show()

source_file_name = df_files["source_file_name"]
data_date        = df_files["data_date"]

In [0]:


stndrd_df_2 = stndrd_df_1.join(df_files, how="cross").select(
    stndrd_df_1["*"],
    df_files["source_file_name"].alias("Source"),
    df_files["data_date"].alias("Data_dt")
)

stndrd_df_2.show()

In [0]:
stndrd_df_2.printSchema()

In [0]:
%sql
create or replace table wh_gb_test.gb_test.customer_region
(
    customer_id int,
    full_name string,
    age int,
    prof string,
    load_date date,
    created_by string,
    source_file_name string,
    file_date date)

In [0]:
stndrd_df_2.write.mode("append").insertInto("wh_gb_test.gb_test.customer_region")

In [0]:
%sql
select * from wh_gb_test.gb_test.customer_region

In [0]:
%sql
truncate table wh_gb_test.gb_test.customer_region;
select * from wh_gb_test.gb_test.customer_region;

In [0]:
%sql


INSERT INTO wh_gb_test.gb_test.customer_region
SELECT
    cid as Customer_id,
    fname || ' ' || lname AS Full_name ,
    age,
    prof,
    current_date()  AS load_date,
    current_user() AS created_by,
    _metadata.file_name AS source_file_name,
    TO_DATE(REGEXP_EXTRACT(_metadata.file_name, '(\\d{8})', 1),'yyyyMMdd') AS file_date
FROM read_files(
    '/Volumes/wh_gb_test/gb_test/my_volume/WH_DATA/*20260620*',
    format    => 'csv',
    header    => 'true'
);